In [10]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, cross_val_score, KFold
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor





file_path_EMG = r"D:\ML project\guided\guided_dataset_X.npy"
dataEMG = np.load(file_path_EMG)

file_path_HAND = r"D:\ML project\guided\guided_dataset_y.npy"
dataHAND = np.load(file_path_HAND)


#We start by doing a filtering of our data. "https://www.sciencedirect.com/science/article/pii/S0021929010000631" this paper is a relevant litterature on the subject
#We learn that most of the noise that occurs when recording our sEMG is located in lower frequencies. And that a high-pass filtering at 10hz is highly beneficial with diminishing performance at 20 and 30 hz 
#while still being beneficial as not much revelant signal information is lost.
#As the choice of the frequencie of the high-pass is muscle dependent and as we are limited by our lack of expertise. We will follow the recommendation of the paper and settle at doing a Butterworth filter with a corner frequency of 20 Hz and a slope of 12 dB/oct 
#To do so we will use the butter and filtfilt functions of the scipy library

#In addition, we also learn from this paper "https://iopscience.iop.org/article/10.1088/1742-6596/1237/3/032008/pdf" that most of the sEMG's energy is concentrated in the 0-500hz range 
#Our sampling frenquencie being of 1024hz, twice the bandwith of the signal, by the Nyquist-Shannon theorem we can efficiently control for aliasing up to the 500hz frequencies.
#Even if according to the first paper most of the noise is located in lower frequencies, since relveant information is under the 500hz threshold, ça mange pas de pain to control for higher frequencies.


cutoff_high = 20       # We select (in hz) the cutoff of our high-pass filter
cutoff_low = 500     # We select (in hz) the cutoff of our low-pass filter
order_high = 2         # We select the number of poles. In the Butterworth filter, each pole correspong to 6db, so we select two to get the recommended 12dp/octave
order_low = 4        # Once again we select the number of poles. In the paper the nbr of decibel/octave for the high-pass filter was 24 dB/oct, so we follow this recommendation.

b_high, a_high = butter(order_high, cutoff_high / (1024 / 2), btype='highpass') #The butter function returns the filter coefficients a (denominator),b (numerator). And take as inupts the order and Wn, the desired cutoff frequency being the frequency at which at which the magnitude response of the filter is 1 / √2
b_low, a_low = butter(order_low, cutoff_low / (1024 / 2), btype='lowpass')

first_filtering = filtfilt(b_high, a_high, dataEMG)  #Here we use the filtfilt function, which controls for distortion. That filters forward the backward the forward then blabla, à compléter, pas encore parfaitement compris. https://dsp.stackexchange.com/questions/9467/what-is-the-advantage-of-matlabs-filtfilt 
final_filtering = filtfilt(b_low, a_low, first_filtering)

missingNO = np.isnan(final_filtering).sum()   #Here we simply check that our dataset doesn't have missing values. Or we would have to take that into account for our pipeline
print("Missing Values:", missingNO)   


n_sessions, n_electrodes, n_samples = final_filtering.shape


#Dois encore copier coller les sources.



Missing Values: 0


In [11]:
#We start by "windowing" our data set by creating overlapping windows of size 500ms with a 250ms step betwwen each windows
#this results in the creation of 919 session per electrode, 6433 per session and a total of 25732

ws = 500                # Size of our windows
step = 250              # Distance between the beginning of each windows 2 by 2
n_samples = 230000      # Total time of a session

#To do so we use a nested loop that will go over the data of each electrodes of each session and add it in a two level dictionnary
emg_windows = {}
hand_windows = {}
n_joints     = dataHAND.shape[1]   
n_samples    = dataEMG.shape[2]


for s in range(n_sessions):
    emg_windows[s] = {}
    hand_windows[s] = {}
    for e in range(n_electrodes):
        emg_windows[s][e] = []
        for start in range(0, n_samples - ws + 1, step):
            win = dataEMG[s, e, start : start + ws]
            emg_windows[s][e].append(win)



    for j in range(n_joints):
        hand_windows[s][j] = []
        for start in range(0, n_samples - ws + 1, step):
             win_hand = dataHAND[s, j, start : start + ws]
             hand_windows[s][j].append(win_hand)


    




In [ ]:
from sklearn.model_selection import LeaveOneGroupOut


windows_list = []
hand_last    = []
groups = []


for s in range(5):
    for w in range(918):
        groups.append(s)

        
        emg_stack = np.stack([emg_windows[s][e][w]
                              for e in range(n_electrodes)],
                             axis=0)
        windows_list.append(emg_stack)


        last_samples = [hand_windows[s][j][w][-1]
            for j in range(51)]
        hand_last.append(last_samples)




X = np.stack(windows_list, axis=0) 
Y = np.array(hand_last)
groups = np.array(groups) 

print("X.shape =", X.shape)
print("Y.shape =", Y.shape)
print("groups.shape =", groups.shape)

print(groups)


logo = LeaveOneGroupOut()
logo.get_n_splits(X, Y, groups)
logo.get_n_splits(groups=groups)
print(logo)
for i, (train_index, test_index) in enumerate(logo.split(X, Y, groups)):
    print(f"Fold {i}:")

    print(f"  Train: index={train_index}, group={groups[train_index]}")

    print(f"  Test:  index={test_index}, group={groups[test_index]}")

In [14]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.sigma = 0.05
        self.features_name = ['MAV', 'RMS', 'VAR', 'STD', 'ZC', 'MPR']

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        self.sigma = np.std(X)
        windows, channels, window_length = X.shape
        features_number = len(self.features_name)
        transformed = np.zeros((windows, channels * features_number))

        for window in range(windows):
            for channel in range(channels):
                features = self.compute_features(X[window, channel, :])
                transformed[window,
                            channel*features_number:(channel+1)*features_number] = features
        return transformed

    def compute_features(self, x):
        features = [
            np.mean(np.abs(x)),
            np.sqrt(np.mean(x**2)),
            np.var(x, ddof=1),
            np.std(x, ddof=1),
            np.sum(np.diff(np.sign(x)) != 0),
            np.sum(np.abs(x) > self.sigma) / len(x)
        ]
        return features

In [18]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import GridSearchCV

def f_regression_multioutput(X, Y):
    fs, ps = zip(*(f_regression(X, Y[:, i]) for i in range(Y.shape[1])))
    fs = np.mean(fs, axis=0)
    ps = np.mean(ps, axis=0)
    return fs, ps

    
pipe = Pipeline([
    ('feat',   FeatureExtractor()),
    ('select', SelectKBest(score_func=f_regression_multioutput)),  
    ('scale',  StandardScaler()),
    ('model',  'passthrough')
])

param_grid = [

  {  
    'select__k': [5, 10, 20],
    'model': [DecisionTreeRegressor(random_state=42)],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_leaf': [1, 3, 5],
  },

  {  
    'select__k': [5, 10, 20],
    'model': [RandomForestRegressor(random_state=42, n_jobs=1)],
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [10, 20],
    'model__max_features': ['sqrt', 'log2'],
  },

  {  
    'select__k': [5, 10, 20],
    'model': [MultiOutputRegressor(SVR(kernel='rbf'))],
    'model__estimator__C': [0.1, 1.0],
    'model__estimator__epsilon': [0.1, 0.2]
  },

  {  
    'select__k': [5, 10, 20],
    'model': [HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1)],
    'model__max_iter': [100, 200],
    'model__learning_rate': [0.01, 0.1]
  }
]

rmse = make_scorer(mean_squared_error, greater_is_better=False, squared=False)

gs = GridSearchCV(
    pipe,
    param_grid,
    cv=logo,                      
    scoring='neg_root_mean_squared_error',
    n_jobs=1,
    verbose=3
)

gs.fit(X, Y, groups=groups)

results = pd.DataFrame(gs.cv_results_)
results['RMSE'] = -results['mean_test_score']
results = results.sort_values('RMSE')
print(results[['params', 'RMSE']].to_string(index=False))


Fitting 5 folds for each of 87 candidates, totalling 435 fits
[CV 1/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=1, select__k=5;, score=-6.255 total time=   4.5s
[CV 2/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=1, select__k=5;, score=-6.766 total time=   4.4s
[CV 3/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=1, select__k=5;, score=-6.943 total time=   4.6s
[CV 4/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=1, select__k=5;, score=-6.280 total time=   4.4s
[CV 5/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=1, select__k=5;, score=-6.504 total time=   4.4s
[CV 1/5] END model=DecisionTreeRegressor(random_state=42), model__max_depth=5, model__min_samples_leaf=1, select__k=10;, score=-6.447 total time=   4.5s
[CV 2/5] END model=Decisi

C:\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
60 fits failed out of a total of 435.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
60 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    test_scores = _score(
        ^^^^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
  File "C:\Python311\Lib\site-packages\sklearn\pipeline.py", line 476, in fit
  File "C:\Python311\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
  File "C:\Python311\Lib\site-packages\sklearn\ensemble\_hist_gradient_boosting\gradient_boost

                                                                                                                                                         params     RMSE
{'model': RandomForestRegressor(n_jobs=1, random_state=42), 'model__max_depth': 20, 'model__max_features': 'log2', 'model__n_estimators': 200, 'select__k': 20} 3.431028
{'model': RandomForestRegressor(n_jobs=1, random_state=42), 'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__n_estimators': 200, 'select__k': 20} 3.431028
{'model': RandomForestRegressor(n_jobs=1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'log2', 'model__n_estimators': 200, 'select__k': 20} 3.462221
{'model': RandomForestRegressor(n_jobs=1, random_state=42), 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__n_estimators': 200, 'select__k': 20} 3.462221
{'model': RandomForestRegressor(n_jobs=1, random_state=42), 'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__n_estimators': 100, 'select__k':